# 23. 93쪽 항목 4 행동 추가 적재 (a04, 사용자 승인 2026-11-05)

- 범위: SchoolAction `kdca_measles_2026:p0093:a04` 1개와 관계 4개(SUPPORTED_BY, FOR_DISEASE, DONE_BY, INSTANCE_OF).
- 끝점(Chunk b04, Disease MEASLES, SchoolRole SCHOOL_TEACHER_IN_CHARGE, ActionConcept HEALTH_EDUCATION)은 기존 노드를 찾기만 한다.
- 안전: 한 트랜잭션, 끝점 개수 1 확인, 관계 쌍과 basis 확인, 행동 나가는 관계 4개 확인, 행동 총 4개 확인.
- 재실행 확인: 같은 행동을 다시 넣으면 제약 오류가 나야 한다.

In [1]:
import sys
sys.path.insert(0, "../../src")
from rag.prevention_legal.load_graph import add_action, ADDED_ACTION_ID
from rag.prevention_legal.neo4j_conn import connect

result = add_action(ADDED_ACTION_ID)
print("추가 결과 (검사 통과):", result)

추가 결과 (검사 통과): {'action_id': 'kdca_measles_2026:p0093:a04', 'concept_id': 'HEALTH_EDUCATION', 'basis': 'EXPLICIT', 'schoolaction_total': 4}


In [2]:
d = connect()
with d.session() as s:
    print("93쪽 행동 개수:", s.run("MATCH (a:SchoolAction) WHERE a.action_id STARTS WITH 'kdca_measles_2026:p0093' RETURN count(a) AS n").single()["n"])
    print("a04 나가는 관계:", s.run("MATCH (a:SchoolAction {action_id: $id})-[r]->() RETURN type(r) AS t, count(r) AS n ORDER BY t", id=ADDED_ACTION_ID).data())
    print("a04 연결 대상:", s.run("MATCH (a:SchoolAction {action_id: $id})-[r]->(x) RETURN type(r) AS t, labels(x)[0] AS label, coalesce(x.chunk_id, x.disease_id, x.role_id, x.concept_id) AS id", id=ADDED_ACTION_ID).data())
d.close()

93쪽 행동 개수: 4
a04 나가는 관계: [{'t': 'DONE_BY', 'n': 1}, {'t': 'FOR_DISEASE', 'n': 1}, {'t': 'INSTANCE_OF', 'n': 1}, {'t': 'SUPPORTED_BY', 'n': 1}]


a04 연결 대상: [{'t': 'SUPPORTED_BY', 'label': 'Chunk', 'id': 'kdca_measles_2026:p0093:b04'}, {'t': 'DONE_BY', 'label': 'SchoolRole', 'id': 'SCHOOL_TEACHER_IN_CHARGE'}, {'t': 'INSTANCE_OF', 'label': 'ActionConcept', 'id': 'HEALTH_EDUCATION'}, {'t': 'FOR_DISEASE', 'label': 'Disease', 'id': 'MEASLES'}]


In [3]:
from neo4j.exceptions import ConstraintError
try:
    add_action(ADDED_ACTION_ID)
    print("재실행이 성공함 (문제)")
except ConstraintError as e:
    print("재실행 실패 (기대):", type(e).__name__)
d = connect()
with d.session() as s:
    print("93쪽 행동 개수 (재실행 후):", s.run("MATCH (a:SchoolAction) WHERE a.action_id STARTS WITH 'kdca_measles_2026:p0093' RETURN count(a) AS n").single()["n"])
d.close()

재실행 실패 (기대): ConstraintError


93쪽 행동 개수 (재실행 후): 4
